In [ ]:
#@title 🚀 **Start Upscaler Studio Mobile Backend** { display-mode: "form" }
import os, sys, time, threading, subprocess, json, base64, zlib, shutil, socket, tempfile, urllib.request
from pathlib import Path
import importlib._bootstrap_external as ibe

t_boot_start = time.time()
print("=" * 64)
print("⚡ UPSCALER STUDIO — ULTRA-FAST HIGH PERFORMANCE BACKEND")
print("=" * 64)

# ─── 0. PARALLEL BACKGROUND CLOUDFLARED DOWNLOAD ───
def _predownload_cloudflared():
    bin_path = os.path.join(tempfile.gettempdir(), "cloudflared.exe" if sys.platform == "win32" else "cloudflared")
    if not os.path.exists(bin_path):
        cf_url = (
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
            if "linux" in sys.platform
            else (
                "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-windows-amd64.exe"
                if "win" in sys.platform
                else "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-darwin-amd64"
            )
        )
        try:
            urllib.request.urlretrieve(cf_url, bin_path)
            if "win" not in sys.platform:
                os.chmod(bin_path, 0o755)
        except Exception:
            pass

threading.Thread(target=_predownload_cloudflared, daemon=True).start()

# ─── 1. FAST DEPENDENCY CHECK & UV ACCELERATION ───
print("\n[1/3] ⚡ Verifying backend dependencies (<5s)...")
deps = [
    ("fastapi", "fastapi>=0.110.0"),
    ("uvicorn", "uvicorn[standard]>=0.28.0"),
    ("multipart", "python-multipart>=0.0.9"),
    ("spandrel", "spandrel>=0.4.0"),
    ("static_ffmpeg", "static-ffmpeg"),
    ("qrcode", "qrcode[pil]"),
]

missing = []
for mod, pkg in deps:
    try:
        __import__(mod)
    except ImportError:
        missing.append(pkg)

if missing:
    print(f"📦 Installing {len(missing)} backend requirements: {', '.join(missing)}...")
    try:
        if shutil.which("uv") is None:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "uv"], timeout=30)
        if shutil.which("uv") is not None:
            subprocess.run(["uv", "pip", "install", "--system", "-q"] + missing, timeout=60)
        else:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + missing, timeout=60)
    except Exception as e:
        print(f"Notice: {e}, falling back to standard pip...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + missing, check=False)

try:
    import static_ffmpeg
    static_ffmpeg.add_paths()
except Exception:
    pass

# ─── 2. SOURCELESS BYTECODE PACK (.PYC ONLY - ZERO .PY FILES ON DISK) ───
print("[2/3] 🔒 Initializing compiled backend bytecode engine (sourceless .pyc)...")
PYC_PACK = json.loads("{\"keep_alive.py\": \"eNqlU0tOwzAQ3ecUo7BJpDYSEmwqZYEKbFhS1pFrTxOrrh3ZTj8gzsABWCBuwa3gCNhOmra0iAWziO2ZeTNv7Be+qJW2YPkCI97tK42EcVn2DqVpFUXFHLEuiOBLLHQjpcuAHG6JMBhFDGdQEGl5wbihSkqktlgpPUedcGlRL4koDLoIMyNwHge9uExHETgrhZoSAScahPDpxhPdYAjX2pVLZvHX2+v758cLjJUgU7hzmOGVxwCh1i9J7YFbMiN4+snr2aRxGkqGz6riAk81b0l7s3qzO3g7g4lqaAXjh+srELysrNgAnwFZEu5YCTzIdoFwtxltGMm4Kfq0JD2s623thm7TH1Erk5wPgOGSU8xjj++o7xtDAevei2uKtYWbsHAlDzvUxJjdXE4OmRFu8qPHS9u3NpZou3c3fz9yHMf3HgReJcOdSmBK6LzUqpEMGMGFkp0CHSAAfZVek9kk7BJXqUSb/yK5AbiwyY9IDdJB1yP38mlvzGZhmKQ9abSNdhS2Y6p6f8r/CLb7U74B53geWA==\", \"tunnel.py\": \"eNq9Wd1u28gVvvdTTCfYDdWIFKXYjiNbKWTF6Rqb2F7ZRoFmA2JEjiTWJIcZkv6JYaBv0AItUGBvtr3oO/R59gW6j9BzhsNf0cmiF0sDNjlzeOb8fucc2g9jIVOyYAnf3d7y8ydfFHciKe7igKVLIcPiWfLiLhHuFU/Lp2wRS+HypHwz5WG89IOSPl1Lzjw/WpULflhuZjII/IUVM5m01yT/mPEk3VpKEZL0LgYORFOcxqkvIhZsbW15fEn8xMFlx4+cLOEG3o+JH6U9Yr4iCyGC8RaB68ZP11p6K/9j6KfpG+f45OiiX+yen86+dc4v5kfTdz3CEpLkDPBKrISnqIHIUsO2dnrljuRpJiMgcEUUcTd1+K1h0OHohWXDz5D2CQrW65HJhNha8qUfec5Scq4UMJKUgRql/GRC9mzbVmrAYy6EMsCEVKS5bmuweJchepXo6s1nEzLcqomrOOTCOJ5kfmSgO9WRJyLi+dupvKvYQFQQB+QhSGglqQeWqHbVQQyiAW/4rcvjlBypP+CxmixIoo/F8HDA6Q4yVMeDqViachklfaJtPSbLQDDUfBfMqeQrouB9ksoPOWtlFaDBlyz8ZeT+cUUYg4E82HovuaUfjbintImVNvrEDzV71tgQUzM/KCUqlQn8iAPnmj0s1AmXjSo+/CWJRJoTs8jLyWMRBMDcT9ReZfHiWgCjqzoLfL1Jghq4qEGhZHMbrxCkA9E4k+7aQA69DRJgHW6+WAuU0FpJkcWG3auHD0qs/ZhHpBuIzFsGTHLPSTNIhMDojOcO51FKX4ubCNzska+1tWclO3KhuBGjdkLPInMlRwIow0mcLQLfJd9cXJydk8v5Wws45mFxlwAqoYs0qln5itGzAnHDJXhJxyvKSC1+yylahN5AfoqbhKJ5NRMeJJxovgtINIibNbwkEgvvrD8JSKECAq0VYAXcez4c0SdLWpP9Hg57oPpgPAv8kt3WTqq84S4xPVCydZrGyXgwWAGUZQuM40HFsnbrDSQPOIB8MgCNAUUHnrZsnchUR5os9Ha3c4V40K31ryCLPjSXRnlAS5TwX+F0j0kQoDDFJujp5C28zG/9JE2Mwv+9ZuakYezAFohZBsgziCpYpg26J+R4SRgUY/cKcwswoZANASFH2JWE0tonN8xPVabDn1Rg3fCTNWGpCCHgQ3HNmwI4ih7rTH05B7XP6aGAqbWptcn3NOcD8tzeRAsFl0nAedysjC2xVAnaaoHPL7euKh8S4MRY0p9//Nu/SQEa2CDUfIrGZ/IOzXVfsHqwLItuCtbsOCx4BHiTPr/mRh5x/cKnncBZJQxq0ZE09QuUdNeh8Aq79oktXux0WAsIJQfAcnlFWpqksp8bqqpW7PQJzWEX+g1qmpEwWZYCbnsQ/PkSqEMRizCDIIHKBmV8j0D9QD9URRoqFLYaZYtnnYmYR0ZDVDi/31jIK+Ck/tbx2dEGDZeyTnN+8fr08qJJlQJETi5kxpvLi2yZ+J/4ZFgtV9bL4aGzp3gvS9R4z8xPU/OPtvnS/PDsewtSvQqc7xFM6Iey85i82GkUceDaSveiwbUu1J0BlQuAf5L3U30CT8kklwGKgMd4KCKlVs9SRc5o+j53n8Y5+A1E0o9bRE/IH5gMSRbXKyT3VpwAkKSYCRD2PEoyWB6w2B9cDwdrzgKAIoCWwL/mZMEBT7gu5Niet5sK1eRJFq24sdeRgw18bDYMH3PJ6zk1z/8anW+oE+l9pflDU2baf/Q1IIBQSib39DLh0pyueJTSMaGXceKygEs0UxYPhpZNHzqZ9DpX1aywCQsq/OG5Co4hAB2OCADT8fhRKSFskAD9nWYJTgAj236cfLPzK67HO+pHgXi7qaFu3CpbN9LduvKxIe3s3lHNWi0uMPinH/753//8pR6HOW8CGS7kmNzzss3ZbBoXmR94AFw+MFs5HkjsQFNyZeStHIo3BrCQkERZunZSccWjcbN3hFBDfqqphGc9JDHJQrAzuacId2NS8SvS6aFovGqca+MJvv+eqmWKZ1RUjXzUGtFMhxtgix7/fkeh6NfHW4wfHrnCg7lDse9VA1AEMex8lE4+l8P+HRY0pfpj3XKzPcnn4o8S2Ve14aMEyfNF67v5DI9uxMI1ZA7wnaABmymmfOe4QkpQBUk0F1AOYjhKE+toPj+dO7PT+fxoduG8a0G0uHUURu+21yVoOhl1IfdHaTHPc6BQscIAjc2QXXFj6ecVoYbI4UopqfYdP2QrpAoCED4QckKfDEfD7eEbKHnYZ5Wrb9RV6wWgqgAfX1iHd9AmHp8ajSOshEEzADR9hMaQpRN6dvJ7ujH95w60Frvb2tXwCk4B1yzIYBjsWR5Xy5Qlru/T3hfG5M2E8fwEeoI7R0cZUDsuk97/mTDVwAmj0lyFIRmaM+B0RaaRJ4UPPSkkJU4KV2CiNAUYeKYmTeam5Ls5QW30lxUo4wol4GCiDsb6MRMBW5SDWCPdsBI1J96NkqK+/QCuQ6AWMQ7GdCoezT6yxnvSIqy58/MY2uCRGx6XS2jCrv6LoFU3f34wJvfudt6atLK9ZKBRMgdW+n2ECEInlPyW7Oz26ls//fAvcnl2Ppu+PZqT84vL18en5HA6+/bo5DU5PienJ2+PT45+Q+vvYK/8938UQzQMxwDMlbwPDVok/Ss5r/kTiqrvMlUEeMQW+CXF4OGCex7cgZeHZsriIj6+hrjoWU2OWgtUCPTqdUxYytXHZ3fpWkSWjvLC599cvHvbL0K/Dm/OIhAuuoTSeouWG7vp1xrxktLmIHbg+deQNXcBn9AQ2jWYBFMRj8lwL77dh2LgYXenl3ZxKccxvRLfkkQEkClPhkej0fOdffpqoyrXT1iKKDURHeHdEXJTmDQmT6b29MXM3idahIUAc4ZAZANRB0+8TiWBwhPl1isy9jzNPF8QFsfEOFi8OkeC7+YHg8WrjjbuYACidQiMuJpId0IRkMcKVwdxtNrPg7Z/n9v4gXY3MdC1TcoerBDoLM8YhA2sRY+8qq1043vpGpTfUQZfc3+1TstHbX8JbXeWFGbUfhoT5bVqqgbLLtW1T8mgqWmH8hgb5cI6DbGQeLwdNXV/Nitc/Vj7pT0d2vsdJbArbDrISg1HGAOtj6xa29E2cBrttfdDdmtqG27v2Zu7GGI5X8xw0dwuIlLbrTUaQPguWegHd2NiQpAF3MwH3j45RBh7x9xz9fwGKPvk6TlfCU4uj5/2yVxAUIs+SViUmIDr/rKt9a2ZrBkM02NiK7eS5+BwIlcLZth99WPt9vY3xkSTBf4K9HEBqbis9ltpU3ebhhP8MomfRM1lwCFsFB/TB+mTkhtZsVhHVTs1R92peZDEkHOtSLYbgWx3xPGO/VUrcIf24cu94X7LMNo0xS59dTDAA78gRx13nuPpauFGC/TCBuRRpkxh5kuwyxnDfBlz6ULG75MAP2VKExi6Kuxsy97jYQVepSydWX0ISkHdI29x8vwaaosLk6nXAUabeugcba6tR028zq0CLiK21itXdDTaUHQPFW0FeIfUqsY2Aawl1nrUEjTukkllWFOo3PqFCC8Pp6PDvS4RLrCm8kDcEKiqdyKTJIbiyInQoI/fvIsWDMZ9bEhULkN7CtU6CO6slsRx24yMrCVfTuh92YM8UKI/X1Colyy6onC4iw3hRH/msqCKqlbAUu8+rb37FBC2CxRVA1Wkm6rC+1ubE7fOEsiBjU0V/v4nFXg6Z2Cpg66WO5jUTJorTC7IY+Ol7fFVnzzZne3Mjl4Q+yu43zvcmb3ZVYf2Nrl9DgVL4MGGXip7jEkE3tkk20izbgodG7txh14l2OO2QvwuGzVr4iNEdRxRzOwCX4f2Xp+8HPXJ6PmwT/DDQfP9jgj9+ccf/kxOz45OyPFJ1ZFOz85agcdagXdfdGMPW4/i8+Z/seqdWad2dTsOuwgKj+5NoTLvdAQh2NBUX17G+QcYqCvBF61Qb6Q/1160Huv9hM4OAztdo+w6er1f8m/U/wGzQBqS\", \"server.py\": \"eNrVPWtz48iN3/Ur+jhVGTKR6Uec1K6utHUeWzPrjcdy/NhNasrFpaSWzTVFcknKj/j8O+4H3R87AP1gN9mSHztJ5ebDjNjsRgNoAA2g0ZxkUeRlzeLqIZsmeS8Rj9eLeKp+/1LlmfqdV+rXr0u+5Oqh4tOS1/pddb2sk1Q91XxRzJNUd66vSx7PkuxKNSyXyaw3L/MFm+ZZze/rNJmwxMBLNi/iLL7ipehaxPW10e8EHsWL+qEA2Kp9L3vos4NkWvfZUVLB3+OiTvIsTnsanbycyqHzuKrjIlFj/R6DPwe84Nms6tPDR+ixd3IoH4Ao+SsvF+LX90AaL+Xv8/OT0f2U04yi6ZQD3wANergo0jyeNVB+4pOzfHrD69bjQVIBBzIORPQCC9NwkcxmKb+LSx5O87JSmO+PT88+61f2kJJXRZ5VXHdGBE5lY5/9cDY+Vk+93jv2Kc+vUs7Okqts4zBj/uEBO89veBawW14m82QaI3ViiivqG+bxsr7eUeCTWVTjAFhK2SFSTdYoHBTWZZxVYpgYXQqGVcZo1dQTwxf5jKdVOIkrrgadj04/Hx7vHUVn53vno7M+O4+rmx/ySZ/9kk8iS4zk6Bks0rUafjA6Of8++jw+GB3B2B/3jg4PItG2Pz4an37eO4HmosynvKoiGhkB3Ba8LE8qXiqQCsjx+PBspEE3MKh3B8qCz5I4Ql3Si3X4ee/TKBr97RwxOzwYjem3NQpWRXPi9PCjnC46HX06PDs//buiSL86s1sujs4PT44OR6cGggiyg92yqKZx2tB4cXK2v3c0Om1x7nzv9NPoPPrrBTydH44MqBIAAe719o6Oxj+NDpCe0fHZ4fj4jA3BsNR+Q3LA/ptaGsKD3ue9vwHs0QWM/GH8AcckWe3nVciz26TMs/AKBngNbnZ3r8+8b7xAgNkfH+9fnJ6Ojs+ji5Oj8d7Bi6B1RyHQHQJKvNw7PY/GP45Ovx/tHUQf/g4CCWC3t9jv4a+dXfUPe4dtnz+wHPTqGqwIm+clWyzTOili4O4kX2azuExAb3/H5glPZ1Uv+rC3/5fR8UF0eAySfrw/ikA7h2RPQ/xr1w/Ca37/ZbC9c9nrRSen4/3R2RnYBuj0SDbGk2vgDVyrIuyQh6tvdFDCIN9K4TU6GOKs+4CSWD2k0vR7T2hkzqARrAhDG8CEtfAzDoyAEbAAfIbGZgKWBsx7zpa3CRi7jKX5VRX0hCmJUjCVQJbeXMIjaPDh9RmQDNIUnY+BVQNW1SV0W7miexeg5tTVCxgsgNzZQjHJskyreM79nd0AJHbG5wxGR4i1wMIP2MZ3OMWA6L5LgBwDP9GKfwDmEiiwkZMgKxsk/U14E/DjPOMCzlWaT+K0DQPfJHOW5bVkJBChH0KAkhR+YCASw0qxH+N0yUdlmZe+t9eswWJZ1TR4whls4vVD6AXPEGZjgwtiztvDxR7TDDvhFu5wbD9NeAZ27YCRbZHbzX6aL2dsHzahHJ78ZQUiADtAfc2h54bcT5YzGDrlQUgSVLPbJKYen8bjT0ejaB+MGCglQJZLvcCJbkGH4glsuYyHVyGoN8ySxpMBwGCGWHzx2lC8S6DG+/v44jT6afShaQ/joqhCsTcBniU5K1kN+/HC67WBDLT/8QVYcumQxM60JIa46lI8aNt9iFpbqa9+RLe4lo286PnQCcJJ++gTXV6KJfM870eCx+L2Tg98k3zOZkpekbtIIyoyunR9hRuKXJLB1GB5egT5HLqK8Qls3qwCqLCGP/x0Ds/VEn5OHuSMIciBpIolsNNj1ximwyW6L5LyATEgmGrFmU+SuYjr6XVntYOQjTNWLadoadgdb+P+vpLzEshqOdnkizhJN7N4wTcLYBLMzKqc+lfSLk3jDHUgrqp8msRoj0gD6uu4VmyLwUtdwrIrtpqKaK+N0siObLStAy06PtflQ/Pyuq4L5QOB/LScolB6mH6gByTZPG86KlxCKUfCWZNCpMfQOAvrvjVxv4O9HmpMjAtd0h46z4V0Q5NndJirPsQlMPqeZKNSKNIi2E5x7mqwuel6bVgzF/OMtkerowdCfJuAtw4bkyfgeX27h+YZdGmIAJnxglZPkiGrl2hp90Mps7pRQ7uXFEOro2pzzhwJZ5wjopM8T/0WHs37wBj/RL84hShMRyoNO9+xQ6HTpk7egam60qoodZTP+qCOKbgsC/zJ62nolGZhwkCKwK7fcrHNoVL6oGGzZAaq1bKQL7NhpwIe2JkJh4CnZBoc+guxcCzAKGO0AsqLMyozRR4V6nrKr+Lpg1Z5YbvmSYnCjoMbC6ksYwjhHJKG9g1BEsAZoMh8Mg8x+/lnJWM//yycNrDm2jg11vOO4Dc4IgJzWLbKaU4aXq20GNAzkgpu6HDD5aAz9NFSB8kFT0iI7LJi52mAquVdN7NrfVFiBy+j8GOcVlyKbQERMazmsOOEmUhjJgPNBDjRPJolV2gbNXjll/Q1NE0Dv4dYdFpHQqB8BJ+XyT9obdbKKLZYxNhDVy4ZevkVEGN1VwiGVZEmoMwMjOF2oKCnQCwNC9hwyHZop6bnL1uXYZrfAd70xhNEeJ3JReftS+2fdfWVEjBMubrTZVmCjAqdlXvBQGU1XqqrejvuaiWL6zqeXsNTLXZgtdFVNSqzr3wNKZ8beZY+BKHSD4k5IApgNH4hjYVNBCcA/iGMwKIMO4INIxnyJSSbtL70P1YLAaycyP34ADGGsG1I84iB9yqqKiPpzr9sbMvjb6zacL2Aks/YnpRAEJOHay2wFi/qC95bg0ATMVi5LR85vKyiaT7jw92t7T5wtQbzNfQuMoUXnw2Uh4jILWDjxzSdZW5VeGEtXChRJgsrlXMRI3GYPYsmDzWvRNwFkaJAczF5SeAuovXo8wd0MP60tYURuyFFMIcPlCwmgR2pAw7TFDxBmb7bmxEpeQbBSl3mKbBbi/o5BaMbGDKyWPWjJCd0FBE98gAapF4B0hvzNLm6rpkgryKdxpRAnKYi58qwnURe7DfIjyjJkjqK/Iqnc9yI76MGJrK9Nuw99gntLsAtRa3VHtiDQNoSEBmF2ZBt2e9XxuAa0XhKikZ4tiy/DisbULZfJ2f/JZ/g1EYiT+GVlxExaBaRj2h4wFKeXUR8N3QxxJ7Zuf08A/cPFr4wSyvx9IoZnOCHbNvl9Z6XS96wu+QpjyuD3bYmr2X3Kn6xrS7qro4biKHGRKyIxiMxWbwWDUmXYwbQw0grVTTV2gfCsVIzfXfCLlil000OfWA5YqO4TB/Y3tmnQ9Zk4BkHn7ucgtyLztshO1xQChcMNlhFoOUX8DHItwMLuMys7W9zApZCOJ065w174GZcJJu325soRpsQeCySWvHnw+jj+HTEpK5RsqoAdcLfuHOaqcPZg3Akd0KJ+u72Hw10yP3cF3mLjSOeXcGCYFTAYR1BMRjYHnAmZgxDfggF/sFBvBvwKmEppvijnmLn2/YUhsVRYgJYC6s2jYt4mtQQ7GP4jbEGckXA3AVPe5llSBma+w2SJljpNAF2kENdAxeA1ZvT62V2g76EBA8cLEp+SzNmZHA5uUi4pShuiTn+FLJ90JbMMNRVmtdKh3AWNMHSdd/kmCzbnOmTGTqPMtz1VbY5LgpogI0NE1ctowzv0P8zeggwjaMSRbCfpxoYTF6AY1PyKQfFgGeezQygqMLY44tXPxTcu2T/MRSRtNeyrHdxUmsUfCdUh1L2dBvSjsl6HCiDVWjBbdUI9hccHIGZ3U20YcdPo3MvCJdFgT5rzyRBAAfMHarg0faoIEOfk/HZeYu6dxDZkqfErsnHqkSikZSXULF6qz5DCuR8A1f5BpD9chm0+BHfkSMFo2QvGjLxLKcMRk6AIeGMo5/keym0Zhvb0EzCVA09CLJzjPA7wO/lEVob/v2GcvGE6XjjFOv8SkUbuZS+gYyKGXCNTBw57F/Kx7bXYVuZBuIVePlg7KWJgLgdbCLXxizOHshskb5X7Y0Jo6qXhreNzOKimqebfqcP/nn0hO+KUfBrvdenvhNk2zvudAo6LUIlAecXaKNLI0nhwKURJ4jERFzbrt/cGVHndZzqEW0Yf2CrzrTai623mta+QsveEfBp2pFtmUffSGmc1/HmxLDuQlvJUkvKUxQTGRmI0VpR4mqaJF4QOAfCZHLsd20WuWd6jbw55G7uCTfE3IOTxXKh92HagvM5e2wv2OYm841QJXhinz+sFEuneG7/cXXnYOWb14prS2zbzTIj2ZxKudlcgMfWFjtwP/Y6kZY2NbDt33LpHND27rQsLq8yVKHLVzItZ4hJySbL6mGgEGo8DzAxn04u2p4RGEdMjYTg5YMNvYoTcFHAOtXpw8tMz863/yrT8w5dtjPplLFNptwylFLhtQHteaZAt8xQKXw9bYS2bOCNOyQdukiD6a5NlmcQVIDqWkA73RbVFbpeknQXUlJAoKN2p4bSnQqlw+6tsDtIfES72RDHC3cCn+V27RxlMwHiPUz2NaBWWip73GsMliOv83XshnCFKSHUMm0gEr/Rtr3GWqnETnVlC5Rzx3A7xS2Jc+nEPMmAFgdEt11RMfp6D/tFLjoern+QMQ6aD37Pp0uKvoQh+R1DpwVCOOggc653eXkDqigyNr3oqliKHApYKPo3/Cv+/UXWT116IMFGuw9rhOs1bCU3gl4kAK+uypDvQYZKkcMXeQ+Z30M8NIS88FvJi7trDEQx19FwGaIC9HA0BaRlDVM7awyqAkPCBAQYIy1YC/LssE0INp5HtgrJumuK3RdxedMAcZgNXO0kW9qBhiyFySkb21TmENoItQaWR2hoOo5PM7KTm+2ghYdIgBOoXXaT5XfgqSqwA/ZoTfPkvRZxRJMq6kIIqONF1Yxvnyhi6QZ/lv1v4C7GGs9TD4GKzw3fzqmhhuQQT2bAWF+dBvGsWpa8JbT+inocu5dRL2PohBWkk9Nhj7Jxqy0VElllHzqCrAzbutJneKI89HRgCO83xHvYbmYxX+TZEFXH5iQl28t20rSrpiLB2PsvV11uszGnyRw8oTjzI7BWA1U1K7fnFcykdw94RopnM5QKkeP8OqlTIOlCFRye1ctZkrMP8RTCrhmDLkAauFNoW4feTogBr0JhqH4EaB4hGJknV1jhgSWy4NbzOIMYBdcnX9bwbzqbxuUM07e4H0XTEjYpUAIwTsS01SDwnBmHsEU+QeuEFGBy4o5PoA/WOFW9CGt0I57dAm1fch1AYzYwR3Oz8rgCZ4rGp4efDo+puvD3nj4R7HsUgWtol8i7MJ7NoiYzKfZxuyhYnmgRlRDhXiVZNdT4mS9NFpCRNl+K1Es1/AIoXZovZCynXgQ9F1ors61mdR3Kdw2bJExVPgids48W9WErVXKH0+UsRrsS34L2Y6WX6RaC8SoqqkrTXXGOGb9NpjzCt7ysE/B9twxzUXIOhC76TLgj8NOGAA0RQsGaC3Ogu+pE4wVhAIpUq5xDoCJqJYsqRGVu9bgFYys9o8UEOpa45fsNbi0/CQ+MXRAEWQ0AReb68VZBQEOZRZUhJTZJ3j54JfA2nYDmGq5bh6StcKv9tkFXv3xSFXGYlgF3JiLDLTYj39joSFIw4BevBi35WSlSnudBDIP1HCTTaVLV4EOVcXbF1Zy4uRXoQKnZVCILE1iULpexXV6qjRNNuFHQ0YzEcEKX4zZpUyy2xqxpLXJxggaZPsV3VG18v1EtZ/nGRVqX8T4WO0xrM5uBgQvBkXVWrVLt1pHLcwe8Ozt9w58Ha15g/Tdl3aWBFpO9f6R/n96Hhm8hti3YaYGB9cMKwuxOSOHeEgysfxzjCVAApFkVCwT31yjLS9JMa7CqgwBHu0jjKfe9//0fBHhv4CSmryqKNtu4iBdN3YXTm9SjVW5JNHRcIv8cVppSGn0jvRGsALZtLqAiMMFand17pGHXeyLBEv2/G7Kdlt/Q5rW3C8Q3OKVtsDv3QprAh0BG40NMjNdNgff03Bw76+dAgA1gBbWPpXE057PwbVGwWGT3lcLuvobwtWS+Nef7R7vBFn7KOpZJVTjkjNoj8ELSJEPOkxVt1UqqcyFfA+pToa9NjAVJzkRtuppI1QbRgqC35NWwE+G/DyTqni79cnvZ7Rmo5FBP090BpaUaCMOA09kKPmixEXrY7BjYUz713OZT3FSwbWd0wx/W2U98jyTTtYbb3fDPLsNJQKQ8Oe7UfC1hQtC28cSJWwa0bXvESSy4mKVlfwwydQegcyd4m1F6DVGeoKPBi8q1sYA6QwQ5BAtgUvpslyogvwlCuybYGGipcPt+0tflejPpY/P76T9VYmrABMLfmMjiEq2RLRQrpLcs8hT8E4dguQjEK1lfXaCkPD29NwgyUOtjNhzc+ZyKDr1VGky6MmhUAugzhGtgMBHfIAdE51X6qu8Ovc7d2b6PYvC+wf6XUQyhCZqEap2/47p+97VYrK/6rXF63HZwNVfottTreCKuVIHbWV9j+vd2J6oWsNLruGLec/x63MCLlGtYMc1TcAXiYgU96rVQnTkvsxy9PblhmaRoQK01tu5pfl2y9JTvH9XPl6wz7maKrIEGIlf/+XLKLQunThoPPQ6Vw3vvBZScgVCbuLkJz0aZa57LIFXado0unovJEG1FkgXXQ2ZYsJfIsVB9QySrZ2cynn2yMZBFG9c8TqkkZEbXqXk2hSh7+EVervbNCtzg0kRaDIzoCK+NdxMiCo4hZrkdWLZQbt5MBF261EZc8nDdqDQGXRVL6OTIShh9jLo/6PuSOsXuYFm+hAg5Dw26AwlipI4KYWArJd9EzLQ2RV4Zi0MVNSLHuiFyVRtUXyPuALx+zQSoSICKGlB+pwW4PjA8WFX+7+imbk253q261vgyrfKc05FR4rWogFYirnQ9v+nL469WKeqrKFdoB7Yyurqi3VgDoW9ogPjh1kKUjwlsl5jiesO60g0AA8Q6fUTjgei0sl/NfePHTt7eozwv6uru39jZsuDlBl3sWYrqqW5/nmHGaeZMpFEHcVPdORn+AQ8GXqk7WbDa8RfxcIkMlXiKZvF0+eSEgwlc8oawryO/EiY1X1SOkyHHqb0nQiBcoAH74m1vfbNVMP/j9wcUI+/8hfnbu7tbBT3twtPO9p/lE8bdlzbAFnx1k3sd68lj/PPW5vbOFvt4cvYb2X4zYLeKp4JNfXa7IohSXHIxpXEsiS3SFb9cMbdgnel543o2Du5zbGqutK/j1N4h2wPnc+MIY+AD6QX+Rn6tcG0H6PNu7MkXGzgtzloxX/zzO3aMs7NTvshv4zRol4N0CRT38deRd0Auz2fwdoQn8M9QwJVeq55/T75iP+6wM3wFuhBXdbDi4N8BED8M4ob3Aet6/e+Tq2t2UvJpgkbVBdglkMqLI4tLPqLvdECDlQvx9OyOLGtcX2+nxUB0QcTJDyaiB8bnZvBoDf7xwzCU+NnZcnwP1BnvhYse4aeAuneXqK9x28m9y68b1Wz/TSQk3XrjbPxt2/vLnOYmDew6IsDvZTz1dEGy5gX6BognXUuTt+/UO328h1XXRjTnyB2XFZ2t0ihy/HwDTiddQ92w2uyAihcpWeM4YX8dj9RFX2uhH3m4qK6e7OwMrUuTiRTY96lcOng7Bm4E9B3PmMrrWD7BYuGwk7rHIcBAgYtYTHUoNGterz8nMo+IpEC8Y/sYdrTr8OQRz2Q5h/gUbQkd9NQ58KC60ddvm3qC+TJNX+Wc7nzb8MUqFMSqHix0EQiB8CFoZz2gpAAP2ckCAvn4iazVN+J+Gp/+JTo4PAVzoz7YhT3w9yyB0Ft6qbJ6GJoAYgN9U3t2pQ5eWv3DxQ0Cwnu4WS1O0/HabVLVUX4jKxK0GvL7WmFMmOBf6EVgFOAJmOGi2MVjcFiF5N5KEOB9OwlBwZL1wvQzY45PDVHVukcwhUEEuiOaccjm3mP7kzpPj2qSJ3mSlxXLOpIXIQwmbTaQBHVY1vZcPbY4Cb0rE2B/1hRAUmkUtkKgh4WA/hwLOurYEK15SO99au51Px5BhSh5gd8L0QiDb3Q38QI0ITk0zW0ldtVaWZWNum6SFopKU8zjY1cRJcVxONZdjTgBIDfdum6LKWZBpHMOu/t3Dd/dcxLl4TTNK1fhpzqA0TwLSXArV8Vp89kN3XmZgRt3swLuq+ouX1xz6a63fKaU/PWFlqvKh+XnBMM6j0Thkm8Kbl+wu8+M5ZObm8WHlsnsvaC27CWrtG5lnrPNf7J2zY9UYIamX5ZiStXHrVNtm51qM0NblLz1nEI7HJqXLf/ZlFm7sZAaLM/Fyx9ig4PtRn8ESvp5PM7QzcN8yZA1UrsuVSFuUzgSLug+rR2oLxb1GskjYAYWCMTMxySVCawpIzQHBVYRaZPKybNqueDNIDudY4x/1RI5i42fsxVS6MdnK25D0E2I9WVHSBrlkZCDrXCwyZ5qYttfwlnwqgK20NUFzQbWlM5OHuTiKe/IrHYxg55eU6xrcRuMRM2bkMUKSprxeGTQMMsIrtruXn+tMA7N9etLefonCZTMCfr/rmLSSAYiKx76bpmQq7eugNPC2/CBEXEIwsDyYdmwZfJFh4/gydp8M0t4bTdYuL/NJWEv+NdZyVf65jq2fCG3m/pzdwaXEgOPYtzTphz2tkwuAhEAfAHPzMR3NVQOkX0DM18PTa+JzXcbDv4A0yCEOZYgelYivOFERFdxgzWZEs2QIl5W/A38oHEGef8mrMBzBvXJHfO8ISR8fWvm/OY1E39reDH7cYYACCZRCyHS+0d73qf34hM8odc6rdCS2+r/ktUqOe6wb1guMfD/0XoJhL/yggmg/9IVE3vB248E//+smLVrf60VE0DfvmJqLwavyiNXxcZcpkxbt5aE47xisddsLrP8LsMo4A2rrYbiZ6aWaf3vteTy6pHkGWZn8fNwKQdPwlOnzNgBYlTlJ7x12ccEgglljWcPDKmTQTwmoY3d/n2gfQUKjkWanjJgLWRskZR9tauDFOhGhI0fj4NlxzjfDihfE+p6J8Kdx0tygiYVEqpPIcC0FB3S51jIL9L0iA8IiPIm5iULWO7NQkpwg6qVx0Mxbr77LbNz+HXCfFOn6NQHk4zP1/vInyEGCQpu0GcqdzjUc9GlCgOtYfMzcCoGSvjmm3YqqpABy49fLxus/Dge8EWBan9lLwhcX86T39/FzA0+ygHy0xrOD13+BN56JfJ/kxz+cn3uUnxtbMV3gFvfonRcATGn7hwJmp+XJG6gxDw+BbLIXr3sN9+eNMtb6NN9A6KnVclyxycV/S8Jq31kvIZuFUKJJrzucwUGqvI1jIHxfzCwrtGixRQ3mzD7rUaF5ocrWt98UQld/EjJcyM7X3MxB0NYUT5YY6nFvjZhjHruW4ESG7IXFnr6meDr44x1HwzsXkQVOa4GV5HoEhZyd2u7XTHX+417Qnu6eIoGzQjr2h3wwjSdMfmP4jY/7K30iRzPCn1F9IltRv7D3lOegmeIdtO6FmO8x4lpdZlERS6oa/3iOjT1+lV6KtVyUk3LZMJ9fOnK+r8D65Ji5QBL9AfJqiwuqmsgo8KLi7Q7beh9cFMQLsIpSoBUsKdn72sy98JLMb4vneDtQOU3KT9Dwo/o61svWgoJqBXMU5uUb1ypQHwoWuGKi6NXrklVOD8K3dynX3WqIT4Vpk40fm1dIl+PPo3t3Nam1t+Kvn0qIvMmjv8fxrjoqFI9nfQzLtIya4Tm16D3f8vmA2s=\", \"models/__init__.py\": \"eNpTVvDNT0nNKVYoSEzOTkxP5eICADbQBag=\", \"models/base.py\": \"eNrtHF1v3Ljx3b+C3T5UAjZynLsGBwN7QHrNFT3kklzi4h4MQ9BquTYbrSTrw44v8H/vDEnxQyIl7cZu0iL74EjicDjfHJLDsF1ZVA1J6rs8ZcURE68N29HuuW3Zpnu+bmmrGpqriiYbll8ebatiR8qkucrYmsjWt/AqGpq7EoC67y/yuyX5O0ubJXnFavj7pmxYkSfZkpy1ZUaPjo7OXr779Z+vX7yK35+9OHv5nqwI4PmD5jVtgk+LtNgBWEM3iyVZbBOWiac0yVOa4ct9CDg2dEtimnOC4zrZ0uD6tOMy+g2/Lgm9oXlzyok5r5tqibRdhOTJj+R1kdPTIwK/xWLxU4JoSZEDx5TU7bpOK7am1V9qgYFkRVFKaZAblvD3KIVecV0UeSxakIYIsHGsbEuuo22bZUEohsFfU93pF/xdR5e0ifPiNmFNEKom+jGlZWMz83JXNr3eZVLXRwO811HZKpycfIHYhfRnIFD35PiOjtIM/iVnSf3hl2KtZHTGeXyCTJJ/vP0XKasipXWNev93sSa3rLkiIGKQNPyTNBSISvKaoeZrkuQbklZFXT/hkhRC3dCM3dDqjsuMD8NVGrOcNXEcKLJqmm2X6g0Gi9nmFMfSHxsgNgYrpL3vLEdZoN32GsqkSnZ13zJ0e5oxIDEGU9gxYBJ0jGN2how9LsBo0YhEH0PJSG4kqAQQ8WA3KmqhXT3bIJpwgNEvNpBgAgDEg93o4gBAXZ/tjkXbaKG5OLbBq+QWNNbQakc3DPQ+uyNaSVtzvUD7gvvxZtHjsCouK7CyU7LNiqQBuKfR0x6fbVUhR1sQAagfSEGonsCLJskERO0B2ZZjg9AmiWuaFvlmDKqs6A2jt/H6+fdzBECrqqhiIKlOLumcDqwG4bY1BUtcF0UGMD8nWT0EUoFyDC4Fd4YQGyeN5ggnhQj/GLFIAHcRmYMrSnk/RavdJc6K9APi7OaQ6N0r+NLHLDiK04wmVSzigtnnJX6Z7hPhxBES8mcAQASv35wRIapeTx3bQZE4O53zKem8i4ov1iD9JG34uK8gVi3tgHmB3J5f6HClEAY8TPGZQU9DA3R87rEwGoG7N33BSNZ7sEs+1uwPujp59oOWiAi8SuL2DDFgOkrKkuabIMg4b9ehRlTRpq1ycq15a/M+d30Se1Pp/vSgNIMMKIFv8XVItkVF1Ct4qqMH23atNckLyFgMbcR0B5OeoHXO3D9NsEVqBgYTDCgyrBPCDJV2ZZuNsJoOjnMpFMB5NIzSGh1Y5ZkG+nRWgDWbyYQ1aKfX63DQDlGrYXlLrYZBJoI/b1YTWHkWUC2law8mE4x3bY4h5CVGt/nUola59Kweo7rxGtQACn+HWhmaGEIgcQPEhul181CdJ2V9VTQiMHFr4UZn26HmBJKfd9zzap58SjQyh0ogdSK/0/V7YJ42TzAMJw1bZ1SmUMhITm+zO9RyTlOI0McVVc9ytq9VVjopU2RbT85kBROzzsaH4pcx45NT4gtMaxanGsNi6YYTg5mQGx9olxAA8MlTmIA9YFZSALDDTMHT0cwUun7mN083yB46aHj0ABlJBAD7iTfSiA6p8cnTaVPc5jAbb+K2yqDXdnGclOz45uQYEtD6+JORlN4fd7AOId9PWYNcjh1qCjznmbaDbtE3aQRWlviVGIPM6LoeVppHwGEXsKwigsPoEBXodfDBDqkwTHvkFOj/gzIUk2R9RyBrrKb14hW0ErKUYV+DMJcE3WJnCeLjWeoiJBQydEOaw/G1Ugx0DjBDIVXRwixrq4U8Cx2dDlHLASqRgdKgC8OlmyQ7XIoukAEHGDgHq7JwSU5CpzD2i6WGvSzeCtW4XFBqTSgtWPyesIZvgOV8a4RrN4oiZ1epetEVM9Ee2ffG2oL3ijeQOUznEaPTujk74GbbrNlhPBnQDMz0C4G/04J4cxlVtyGiLKr78M0lHsQlepmC+eoa0gyYzig6Zr5VE+t9Qm3DuCtxul9Cam5rYNzuhdTehrJ3YrQ3QnoDCK/uYzf83bP+6AAVnBtMj7AiZ1VLPYuZ4eYGfw56iy3gY5ocJfsZJA1FM7nhYlN0gzsvlpf01kL9bRy+WIdugz0ILh1lR3xwbTxiwwZNaIl7hxenB61shjaj9G7toI0a0hJiv06Q3LbKw6QxcjiV3nltzRDO0m1qPWxy6WqlG9oiwineDFRH+xnzDK/Yy9a5ZU2nWmMTREfEt9nh8RImTKWdYr7fx/EN21YhABTR7h4uBjzkZPI4MUC76WQc8MT4HtY/zYgs+/r9cMaYOf/sO6l8rutrMr65/+O6v3DUz/Z/w270Bit3pzkxANGDRZknVfaqaXC89nmZw6EO5YoVU+6uQ43TN3XM8U3LMxx4lmfO29aYTFE9/LgPSbHDT8O9GndP88zSc7bZNxhnqPlv7JzNDjxfeA/NJf2xnTIwEiVbeVxnH771Y4IE9gQG05FUXMA6l5htpVX7FpnmCcjf0MdrQrdbluL5SHZHbq9YRnkhC5ApMEWkO5fBwZEVPJ9BkKukJmtKc71vGDnHEUix956nWnPylKmd4P5yyjEGqqNXWDCGZn604KVH6GtF26yeRs/+GhqHyuWGV4lIM/eV+fQqP4zKnEG9h1EG1K/z0E26vkN/g/lw8G2yhkPW++x5jnzwHDKUdycZoGfH8kAcgxE103N+gg4oDMMhBkuEiAb7LlFegdXk6muK2O5qtrh6gga6DppO+OiCNTKVYR9oDAdzkaE3f6RxVepg+ZR+6yWfohxlJP38tl8/mn/2lTkr67RU6dWssajyZqievJNXLqiAtEuqD7FKFmTxiFUEB95/qL9/7koT8Bik8DrKoo7wOaIfWd3UgdEcusO3s9aj+wG2Ns9Y/sFC5ASX9R1v3ntKOwZlqXOimcndyuR1bF2pTyFGwyMPjVPBxQkyM3ecESBGah9m1j3MqHn4+sOBm+y9V5sPUOWwR0zoHFfGBFWpKStWHbnwF0wLJpdUD7pE7C/L5NOjetLIemrmsuvrnmv1MstzsjbXcEVtSVcGKTRVXz7YPObJqXo+E86x3Eexf1k8NG20SjSPaba+iqc51U7/K/aqBLmHxcprJr8U61+THNBUzpsmFb2ELKe645WOGOSFua5RGBCfwBgAG6zReGaUVA3bJmlDcC2at6XnYonwC9V1WNj/3XOcZh1XOqwLI/KCDK4LP90fVPs+IAG6DL71MA/vbzRF3N2J0cTJBeuAMOWfBgaLK1XS/8k4yRd3BXAY33L9y1/A4VvC5p0lIyNscxrTjyWrrLik7ujg/bcI/3wfhNEV/Xh+evL8wgTDmVKgDiwDFxhWrloWJZCVp35FS2alH5e9+10oo5X4x25yiWfl+qi77XNlAG39XLB1IS4x9cO2Uz+e6oERmz134bkY3puSW1BIiLJKvDqHJin82bgXxm1BmUrnpzNnPDmSEgPe0AsE8rB/GDHkTBLjtt6OtANPL5W9TkqV0+yiYbBtInFi9JSPzOB9qFDhDAMbGcAVH5ZEzcgAFMnTG19WYHRw7DA6o5bXdHyTtfcgRWkVshwDWz//nxGSHBvPcp912l+cu6xzTNV5009danDKzjBkeGt31An2lfM9wl9UFqXTAZai5NLcSNKygNSB3dAYUhlxkBWnkGYYpZcsb/YMIu0uOBGpi+VX0U2StbQOQu6CvXKdXgZsBB17InOTlRe3HheQ/bprbzp7MS4mjTLF+UAR8mOS3LgNxXliDd0BS6FzEwtDgL/ySoYfO+se3cVVFnZLngy7/ujJr/x7VlI43cUkzRUaEjAdht6ukrt5E+Jek6MY3INbWvJAQ3hrCVub2taRx1FGtSbEC5j2E+iUY3Yk2gzIDB+p7kx1wFwZ494LnjWjWIy8iavA2D0UH5x3k92scswH7J+W4ED4HyEEHMGolZTdlm04bhdltyP7gNuwMiBlNA+kcHEJhma2E6svYEMvxWDk/wAu3rx/\", \"models/denoiser.py\": \"eNq9GGtv27b2u34FYeBi1ObIdtYWQwAPt2vSIkDaFOldvwyDQEuUzUWidEnKjlv0v+8cknraThPs4gaxLR3yvJ+kKKpSGWJ4UWUi54Hw7xvFWSrkugWIggeZKgtSMbPJxYr4hY/w6hbMvgKEBv5a7qfkUiQmaEgk2/PmUdZFtSdME1m1DEqVbAYvkZR2iwwc/aJMea6jgqeCxbURuW540YDAHzNlIZI4LXcyL1kaoz5Tu8ILYeJKlWvFtfYgmQA5APKt4Lt49eqFg2dCslx84fFWpLyM+QMycEtCZlzFwJcDbcUKT1zoWBRszXv8yopLTyBhlanVIXynhOHKge2zIwlEgFsYBJdXn6/fXJGlN0UKYiacTpI6ZRMiMg/G1wgEYFsmcrbKOQ0J2IiTSVLVkzB4+3HxqqVxZHMQvL++u7u9+wSb/rCyTDbGVPpiNtvU6zW4M2MJj5JyJmsl9mU5QxWsI9LVWSGUKtUMjFrmWz4rmJAz56TZ4uHsNfAUW67OXksjboTkPajQJm7BUWU2k+mQ+1qYTb0CxsXsplwzybZgrVmLfPZRlX/xxJxdclkKzVGInDMNPBr3z7aLaP44xz+D4PPrm+vL+PLqw+31p6v4/e3l1Q3aAuLtC5eaG/p1sniImVclZkAhRwqTb2C7IMmZ1uTT3ed37z5w86YsKpYYKmX0vkzrnIcXVqmUZySOhYQYjKnmeTbF+AdAnGyWP7uXsja9t4wzs3zxi3tJSrldLl5NSV3phOV8uZgSYBNDuvHlBAI4B087Tvin64orGkYtx7BbAt5RQxK0bB6HG1ZlusfFRosbMN2YCO6JWAWxkKK6b4DIOTw1WnVaTAmoBAIvwscpfLzjN79bAhXDPABv62VDJMSAb1QmyyXxWrtQB2yLLGSVQ7Au/6Nq3uOWlYrEkLtEMbnmtFX/jJz3rPYEzZw6jyn2/1fuySK7+OoL3YYmGGjHVOoj86FnFECCQHgYmDJne67QnC3roREdjt1F4bkTVXGogxLXIXHi1OVtnJfJPRaopt1ENwCAaGs32HoCOz6UkgMiCrzmZrhM7Xd8z/cXRBsFu08krddtnZcrlpMhEVfLM9LSIrI0qOixEtHprBhQIJ9ZXvMrrIY0m/wuIQexbfCUeBbKkSU/fG3Jf/shmngv7KDYkYFJOvog0cgWQlvJ0CBD03sLj9RqtzjOqUD7YNOmTc+PwKD4DEs0DMmMTAYU9KQlseNivTE6xhEAqHQUZ2jxk5W2xT/WoanvQNMB9SkpoJKs9obr5WIez+fzsNPEqFHQ+TFAV0ymqq9zqzdI26xiWeP5DfC3ddIJAvOFkwbih/YF6SUbf0h4ZciV/RGlHFWPtG20SPKQEKjEKvQuQ+Sl7dCd0qXM98u3DHI+PCSrU/QRndjyoWNeMMAcAi1gVIsS15CAwEGL+kf95ygXZ0htGIwyKYx9FMQ56ge/fxSYTThFpqRu9gkjvmV5r/NAKuA8c3GUqkPesDyjvUg5KCNdqj+SMcG/nR/twAeDolsAuraTN5txEHSVZ0rwOV6t1QUMtJFMmVJsH5KzX3uvTmxRrGO1XqEZtudRsgWP5FA0GvypBb+5vbm9i397d3d+9+43p79pg8uGqp2fqScWRtDxi9pwej4lc6zuUQbOgKYd1VL/t+b8C6dzTOzzly+jeUevZ+vgqIHdLm/Urrq7omtaUCwrgMJD1HFrZYJ2g2J1MsHQVFTUyQk5ABCnjSPnXTKwDvU8fnQKhBHT2CwpGLcW0vwS9s0GJjsH04W+WTTestN14y43aj/FX16cg9Gfdk2xoTVtQaDgKmU2KC5Ox0vYITRzXWd4jAcNx5Dly8X5CFyxFNPRAhst4WyTwNGm5fVXuaLwmRJXHC7sQewPKElTPJf9aVXtGggMF2BgX9QBLeoAQdc5sCUusb9SR9RVH7sGxedUx22DBCk+q/k4HN99OgJRcY97QQYujbYD0RRKM3ad8t7NRy7/4eTVzA3tIUv1IYrtYhfQPfFmJJvgQiPFV7QHfGKRfouK6sWkOyUeR/aI6QlMBCJ9iFtIEDzKGt6YHm3rhfI6DDodpCc2fqQAZVbZEy16HcYmOhplcU/B1D0cP6F85RC2NDwyKxyU4EdGq15VBUEGh17axUs4bsxrX+pEgRNef+dwY2b3wmxzONcMFMoYJiGFIQtPsFATCdJ1IUysSDhWjdFH6jZ1C7IWxDtaIkCacKzKcxxeyfVkrCF6z9nCBqMdETxZKGFeoJDAlG1HT7cS2djWYw8fNcpb+4tGsQwazdJnGQap1lVq49JfmdDFfB7N3dEB/vHRvh9eotBGi/BEPELzz+EAlA6UPxqbLQwSGUp2paFslgbSzvoJ3nYixUFxYyco9MvBrcvRcPM3L9beO/ezGWK7HU0qet4Dbr1kcKmI12PYLOEnwq9eurkGIdIHWJ93eLsN+IpgvboYxwlaa8fg8C4ykL3WmL2PeP9Ujh+zpYf5Foi5yarIZmZ4LFph6yHjFey/D56TTPZlyGB050X7XrGDUN89Q9TOoD8tySIYy92zNyxjNnWQf5GXCJ0jlPajifwKQDgfDLH7O454YHCt6Pptiz4O1i5MukwfVNQB4/mJct4k+rUtdzZaYZ6WhkGzTckXrlw5tKXRcR4nfS/hx+cZvGzlg3ZjZwHdBdgzmgzeXjyuBBaA/l1GTzLbXvNh68N2fvLo66LY3gAODwy+659EdOs93OC7qh+ePrNmlHikVjc7agmD0T09OFXefrLXB0PEimkdPMnc/TwfCHj0PrsrbV2BnHZDDYyNENFcbXnM6lSUfsgCAZbwCYPvVPWWjnfuP42zA4M/2eiPGv57xj9wwP8w5v8GETG+5Q==\", \"models/depth.py\": \"eNrFWW1v2zgS/q5fQQhYROoqapLbLha+8+LSJukZaJsg7QY4BIFAS5TNs95OpJy4Qf77zZCURMl2kt4ecEFtS+TMcDgvzwxZnldlLYlkeZXyjDncvC9rRhNeLLoBnjMnrcucVFQuMz4nZuIKXvWE3FTA0I6fFpuAnPFYOq2IeH3SPhZNXm0IFaSougXKOl46WlJeJiwTYc4STqNG8ky0Uj2HwB/LuYyqulzUTIhADxUxMMEgW3N2H81//UWPp7ygGf/OojVPWBmxBxSjp7iIeE4XLMKN66GyYoWhjGklm3p7/L7mktV6WD1HaU1zFAJifcc5O7+ZfTgnU72hMAF9Yua5cZNQl/DUDONrCArQNeUZnWfM8wlsmRE3rhrXdy6ujn/tZOwgxnWuvv0j+nx5dv7pK1A+KoXchFVyGdFiI5fgi2h9EomcZpk7MVOH7dTbM/V6al4Pb04OvyLl4TJ1g32y5lSwV4l6D4Ra0pPj3Jx+mp1FWt8Pl58urz+fXqHK4OrvrBBMeo8uL1JWF6UbEBesPlcPi5puREwzhi9rXvOEC/cJth5pzbIyXqGN2lANP8EAmAYmaMISQ6VjaaJC8VbIOrCeQOW7OzQeaOkkLCULJm02T31HK7aZEGAAyr0G9snh7yPJEx1lKemkkKKUhBfE9p2mwr+acgiAG5o17Lyuy9pL3T8K0VQYsSwhamUtixw8djKfDkKIFx2OHAgs4/SiB0qAAjtN1FErZRj4YTfhbSfqzul4lmnEEzCQvTWbsKWT9Wa4kIaOmhYiLeuc1V2qnzaynGF+XtVlDHleollh7DMKvShrFXXnAoCJSl4WnVD2EMMMmSkpypAINGy0PWXr66ZAXNPWHsyrBFArENYtgS5gRQJQsyEHtsoHgCXGt0JCNIC3gFouGZnTeAUsoTuQ7utdM2e/Wap202DTbUuEyI9gB1rwgiWeMr8/kKAjZfqczV4tJpSlp4HND9maZt6QBsILAWuyZULNvKRZanEYB52rHzTrq9yTuheAfmBZWRIMSnI6258SxHtUW3nyJ+SRPbnbBtdxjbnvdqYGaKv6WHOVPBhTv/AOkts3tUon64UkgUU0gTNKLjOqoYdWVbYxIuIyg8Ciladfm98mUCjDIqE1QGJA2ukOlFr0VCDUU3YA1DK0+LMTkX8QiDqZB4/towVF9qLTqY3lk7EVoC0I47X8gOTdhgM1rHSLPl6f/vPk/cdrHUExiIyUWFPzVKq2Fpj0fLCnaPbl4vz6y2XQE+rqMiL79sf1e5uoLTYjspvZ9exs9lUTavdbe1AOVLv4bHku6BS+bQ1yBzZy/q5Lu9IbAIWpsIEkwVBQg200gSwdSgkUl8moxARENx/zRf0nQ6RfA7HsS1kwAthj5UYbOxYhEugk2Z4cR9Mglw2y6sQ10KmT4a9g0bQR2EcKyPZCkhTwFFEUQ+v/Fs42Gvd7vLXMoytci7k2jbaQKZU8X0T1Yg4kg7DvvGjHPYT8yfXH92bbRdVIAXzdkp5qXsXUiAxMLEYSeqqyFlO3ku6I93E1IWsLywnUL7IKyBqtqKlC6Ghz4flPzk5gH4jy1gbakW4dJnAAYJjtOrRT8Kj8y4lua9fPr6Wb7EYa4br5evNGk/nGBQzt2cIsUBn6cDTTKj5mMGH9+sDU9oQIgbOR3GzJ03buQmQZkHvV1BpPhmJJK3Y7Obmz1cdy1xqogPrbFGqMZoAEcKyoyoxK1rciozXDphD/bhj7zrxjv8crAeebqYcKWIO4h6k753EDH7cfh9PQooAKAzuDILmg4B096YetbF/7Dn7hMALf6rTmmX0aWOK4D00GaAKvXr9D37cp6cOAkj5sU4qKorie4bBfpnUn5jrwcwEnOogbryPwEYf2zNIH//X+hsxKmhj8XZTFoZZjYoysETyEBUFK479NyTE7tLKjBX3YCyjzndWl8IxfiMqOKQw34OnftCBMjTF3Adij+eOMV55lLNsqPnmrdAjIUXgUkOPwyN+lhWfJfENO3r0DupAKVMXrVbEr2bNtSF9XfNO1GDQyDP8q5x58AlJRSAMxrlaq9vQ5qLI7wusE0BTYwn5AV/qugcAi5mmZIRzR4DRtpvBk2FU2v4dgdc7ZZtNQHDxzivNtKAIAR8DHGw6vvSBBQfgMU56PXjCyDBi5I/YwXyElKAG1TEy/1Q0LoAHmQkblSr2aFWO1TzROf7NQ2yM1vcdFOtxTyr0lqYsTWodHNCJ8sPEN8+oXt78B2c36HBsOomiFS+omRrLWWWhXo49Rf3B+MdmKEuA4VKurGowSMLXnDxt9pMlpvYpiCk0QnpxGRwsdlgNUM93HdP9p3XdsXQb3PF4fZCNVoI6assxzvFGwKcenHaTdqidbe0rVgcWDbgMvbfDggnJ13BOlUosoz+y4z2goKqDg/uYwQLXsFB3t7tUxUBULd7xjdKi2jQpND0PAyERwaxXswNjMhSrWxdjtO83Un+/UEu3pDhLjR2yFYpsqUdFqbgi94yONkvofPqr37TtDr9+IvydOy7zKGFTkgQV2xmw3BrkNjXolAtibhFxUPQK83fNEQnVYMr5YKs9s3T7ujEFzA4mOjO71z3LIrSnaFDVrD1azMkSnKNZE7EzgJ8QvKw11S8MTrONHPd/9EtxFEMIm42BBa91TLiOegu6NwKx+JgD25f4uW5oxc/DBhKVVqNLV3xWyQLq98BzoV85/lWFq1X05NroO9lpHdcKHPvNH92CtlX+GtsIZb8ZyAkxjlvUjP5F3OHqEo54dYuR3GKRFMuS2KXa4ZXC5rst5xz6O4D52bAwYoO9g6aM96N9CwExBowpisHEh1a0UwUZKQaeCUb32GA0sJHjhfsnEJ1SFLu5+oCYNW7Zdm0BcYBYiWJqpUpwNKyUWfnME3i4nOrgzRoWdkMBl+oO9jHre4nVe3PrWFSRQGwR5BsVbiqbIeLHavuW7/Ko67dHlJhXCeZW57fQfKLjz/3V6xOtxM+gboADPU4LVaxbRJuGlacdAgSl8fOcFsO/kGOf+2TjbMvirjf6s4V8y/pYD/ocx/x9JK7xf\", \"models/media_utils.py\": \"eNrFGmtz28bxu37Flf0CxBBEK7LHZSK3HJmO1bEsVZadthoN5ggcRUR4BQ+SiMb/vbt7d8ABBEWl6Uw5I5E47O3u7XsXCOMszUuWFgeh/FWGsdC/57wQr0/0VVHNszz1RdHCLnPBgzC51wtVHkXh3M3Fr5UoyoNFnsYs4+USFpkCuYJLeaOsM9iq16dJ7bAzHkV8HgmHfQyL0mGXWRmmCY8cdlNlkTjQdPzVsf6ZVHFWM16wJGvYSnN/eXBwcH4x/Wnmzf5585mdMmvkZsn9yGEj95dMfwv5Yy3mGf2Yx9nIPvh6/m52aeyLsxO6G6cr+f0gv/kq1NtjeeNkBdsPvCBdJ1HKAy9K/YdiwoLQL2+LMndaibkf4dYd4H/81of37iueB3CrC2wB5oNALJh3L8ruFutB1BMGBGx2+La3bXLA4LMOyyUbpCPv4ydcMMDDkrRkYdIHbsHw07t5C/vuhhjW8LkoqzwZ3KYOFRZeGPN74S3CSFhoM+2B5mkaSfoKD9oQwdhuUS0W4caN0rXILRsZb7WuUMf8QXhiJRIvCOPCWocBIg8TsK+lCO+XJV0QJbKyW7oF/+4k0dFoNEuKKhcsS4uwDFeCITbQaiySAuyzYIs0Z2Djm+PXJ0d1tTo5HmdMJH5KsoD9hIdYWIOYYr6xjh1mWUDDytMqCSRTts1esJc2Ozpixzb7Dv61+5bD++QBdmxU0pJ0HYVHCYWXaRz6rUJI7HGY52kOBovehwZ757AAHNmT+kCxOywOE29el6IgsQFbL8fHJ2MSHwI0MnunULM4DUTE1sRpwQq+EFENPso4KwU6LM9rhuQZTwLFF4SBGtjPIu4L4iBMOEaCRpZoP0B6wBWAa6vhGSJRkUYrYdm23fpB1LiFMvsWXmzg5AUYEvLSLhclLy0wttIrwt8Ee3tqSKHjGErkzc6D5m6LLOO5SEqIIkGYW/KiOL3JK4h6RN5LH+iydZ8yzmgrnLhFg0fxpPkbB5YLYA4jt6RgpnFEHEAE6nfShNXb2cYX9Bu991OaiJZfNGkI5+hR2iw6Jy3zurugZKl5bUS5DWUeya2SKEzMWNGK8ldgqptR3Gv5bcEyei8PRF6cPo6+FCI/nN6DJEcTNrpIfwshjxy9csfM+jlMwEAK9ukGDNUd/8Bg4fXJD2zz+sQefdsmSxbSowqXaSYSC64dSpBpVZ6+PAajh7wDJpY5jO7rQ0EuWM9HdBcgvcWwCNZLNHrU9PB9/PjLKkFDRyIuBlcLvQ1d/NVre+cuUANGcdq8Gzd+5oDzYScEMe+u87AUFiGzn6Vxcp5m9dm+01AtXOX6hkAbCx8+9Zbf9QFa8wdpXlcJanGGl9aiiVQikHGoTFNWxBCDGNUpj6D+b6MuXUGOwxr/QVUPqLFDVXRuLyCkQZT7Qz405ILPdzDjHJefSRa7sWUcqr42nfOwEH0pvucguwADuw7ITcyHtcc2RCU8Ft8m7LEVDkpXpiV0I28VBiL1fJ6BTgUku6zSGUhXBDJPQxXofkXYMwnqsAWQlbm7/ddm8UtAzgg5U8jJUlc8CgNeCvb+6rNjZHWH7qLTYUkKtgB8Mx/SbtmkIUADiu3zYbAspa79kWduWCAXIjB1OijOLwlRBdGhTBihlMyD6KgAMqhImaIUybSyArgiWVhIE5KkhTyeTa+8q+vLKw8OajOwSYiHHQYT5A8ME/0dkBAMIvvxFEH11VsG+d4dt/xLet/DGi2Vackjj8RVqKplTJrYwcz19GLmnV1++XQjmWpSdUAZb8/Gn8/f3XxQG2mfLIn2b/wwO//pw425E6QgieJ5cV2hwsv2tIgyF5GA3siscLd1ODpPyLSUyRnlYiBK4ZcicEedQg0wOyhMpyNCRzKli9VtR6H4nFs5X3tp1a3VANlEO8X+urex458J404XMmBkutQeFMjCmCpfuNeeuFcDO2ZNu12ca/6kcBZplfu+6WaStCdvWN+NoD9bKUlKWWwDU1loSsh2FGIlcavLmd3xC4n1+b7bhkL0pRCkA4lPWoGWDS/ZoyT5bSN/LL/1zGGttDBYv9NNaSFQu5eWBJ6wLR1KkPk93AP3TgKe5xw67ZLnWDuvlUGoy2VrElgNNmq/FpS7URxCYJ4sUhUPY176S/Bz0yYMSxcb7pdR3egfdIsdUMOUWyx5Jm4nx3eNB7I/nTbckRMaC0ujbE+AlI6+559uZtfe9Ho2RRTWGmJUF0VzvbSZiEBj7a6P009n/778fNKGM82cQp7T4a1mGYxFI28FZzuKozSiTuVUXplWqQqpBo/OedIkvCwXq1CsvfnrE2uH0jRI68rAom6+mqIe+7bJs6XdxLpemFNGSE0BVQCKttGItm1ohzGoTpfsiK11z1Uuq3i+W5adve0ZtyXaVbTEnT44bF4tFHa0O5SlpWc8RNpht5QKcPvFz9fnNzPv71ezn7x/fJl+PL/5F9B5M77r+Hv6sFsO6lqOxlxQliIJXNhuICT5qlwcvmmKGhGHEHTy9B5OX1i/pHPtlWGw0f5nBHy1BHVzXnoYVpoQrmVDcKZp9DxWRDwrRKCTrzt+JdsWF/9BfX5oIJcH96scAqqsGzRroEOFSMumk9nfdq0l3sr4HejDFm+bNEXJqbdp9h4ZDGuWWvDMR3uH7gGaIIBxNDAYkcm0SRenIQSsbBF9f9Kjr+uWloReAVW5VYaZrVUfQBjq66dqzbSDuJ0h1+7o0NYmshJ5uKhVQoccBZWd1Sby3hQMgulXgofwigNPoUsM2S6A3xRgxsmhiLOyllUslgsF86FqSaJaJm3IQFjEgLUnolflNuFa+UO3u4NgMdTY/UiDoC3Hec9B5E8Vy5iZKSOraGE2NXsK50EyalEpCSly1TpvzSNRpNDUlyQiCU6iK8mX2lVXdq66Ttxq3boloZrS5lXixSnk/zQXgbdYxJm4t/w46EzXwMKMccw0qfUgpj+ObIM6puMKlPn+PWI0pvJqoKBo4mgdsANviS8iGUJJ8WBdLAszgXFzgdkaRIMTsUbpiA64aBG7VzTdAN6RjwCHH8bNd7Ovn758/Ei3oJszb12dX83AQ8SmNAZaHQXvmoGA4pH5vjYg1fIS7IVi6CgsPHU6EUCwJxMYaJV3tsnIpvsQRtGO5pju+2kcV0noQxCwmtGP7pueGgnsoImN9Nb0YqB5AGvPpdKaM7J5zaoCConeMGLwgJ5WCKhy50HG7vGrockbeQftkr8xikGUVQghAIyG5yGG7m8kidkmC8H+t/nzUzhtUkmf3S07MATiA6qACOf7hZHmnjz/U8p9vmL3K7WjUFKkCgAUJSBuqKAuQ7UlawjZiKguTUba7pTDaQNNHxISCNjASni8CsJ0QkEMdEwDZJ21hmOKc7A9pafva3EoMxVypnIJNC8fXPN5xsXViUwccJIo9EOImsgAW4JjRhhtaJaCeQllisN6wn2l2C30qSHfsAU0SDh+KVJZ48UiCHECIzOfdvsoLUqJ5AwTV8GqDIjUIPIcm6pWPCZKME4z5Lmdg6p8ojjppDS91stqp92ieLDLNw6QiwRcpEnIcJA4LAqUDpCgfOzqKQ1WkB7EVFDObUNgJPMEPk08rOk/PmI0mldb6vhO4qBEIzbCr3BC28ktuxNIN75uu1NORZzhypDJZPBXUzNVoKhnFr3w3gsigKwbQ3BhOI7oh4S7Eidly1P4UwLsVwo9x+hwA9s9P81qOFgj9xeG3Bv5k7xbZ3QGIGJOD4vHk9VkTCpSCy8nfDL+62hoiz+hJ8bqESFtysKNt4hL/K08bHBnnK4WEb8vEO7FAso1qtx3EOEIhcckCsUSa8NiCBgNqvEeu3v/rq9GT2quySamwZFQ7X7q1puwcx9vh8w/QzxVIWQ6PQM9gm8T+ALKqjn3H6AsEAkEhyr3BSUKHiY4M6F4Ywn33oX4VhVHX9N8HhZHV2cXNjo9uvYcN2QheCBYfYpBazvzANuc+/tM4Xnm8F+axB8zi99pGh3zgJMTobm8fPmX44d91rLPYrat5jmWA4wYnWCnL9NaSlKp8r1OuyV/nvy//PB/51rq9PbBwfNcS2LxwFmwHpBXEGrzMLPs28OT8XjSoz84upRdRVxtMGkt5CDTgkRZskdN+5s9YY8GtebBg5FihxraViyTfYyM3iNsv79V7GDJcJ+HZd0tN0wutkpCFF0v608G6nAJsfXI7KnHZL1HYzKJNUfdX942oDsYG6xu202Dj/f2PdYbLFr/Ru9NuWECXSGUg8LDFzcAMb2eg4teieKXkwtZyA7PKekWFj2RoC3eIpk0r3bd3raQ0ACbF3KfzyPRvFoi1wgJlmN6/dXLY+NOxoNmw2tV5LZ4e6Uuts7yjZTmoIz7eQr9cwrmFPGM3kpDxPTgkq/SMGBfr6cX7PLyAivNCEe+8uiFLC+nVZnGYIT6vRXwOdhMxTI9R4bchfgYHqFAFGdf3k2x2D1MF4exiNO8ZvQgtOiWq3vmtzhsEklJMsa3r7SUpDUOtdZDbkGD3w4qLOTX/dXJrjaxq+jOeLv/LgGcB7/wTDglJlXjEdXPPjy2Aqeoyd8E6MeyTBQO+952WFDWmTgFiArU/6ZHkVTobfAdvjXkjc4RD9XrUubiwGasF63lszZ3H+5DaVvjqzNQ2txje0nIBiIOAm76gJsdz/vThTzOBgS2k3ETGNmv+8CD0GHiyZHwRs1vJa3DxsEcNrZ37YSeR+4LE7WvJ7AXBpq1/TQDtcFA/TsYqA0G6qcYWNrDIiAI0IThb7ctVxNNxjFkpRc3d7sxpmTEPSdRtHZwggbeakNKdMhFTPh9OkDB70XSakBK8TlEn5I7CrtBMogFAm9mnLUxu1bGT3NN+/XZO8heMKuVy6EpU3sfJ7XmpDY5qZ/JSd3lpDY5qTuc1Lv1D6Hv1gCcNAgc8yTN8uZOx39Yud39NpnB1qRl2OlIrr2xGUR0N9TmS5b79ZJZ0g28IQXJZ0TTmwWTSXCEkRALaGE3b/RiMur48lv28vjN8GtTVL/4VcDdsPD4CqpFrDh2vj7VgtNkxvO5vxQD48Jenu1c4iu3T882e2XWfwCtPPaD\", \"models/rife.py\": \"eNrFXHtv4ziS/9+fgvBgN9KMotjupDHwwYPr7iQ9wXV3etOP2TkjEBSJjrWRJa0eTtxBgPsQ9wnvk2xVkRJJSXaSedw2dhyLIovFYj1+VaQ3WmVpXrKI/sTRlVuVUTyIZGtSZDwo68e0qL8Vm+ZryVfZIop587zMuR9GyXXTEK3Uy03Gm5HfIjFwkacrlvnlEqaXjLCP8ChewBAgVre/SjYOe+PHsX8Vc4cdR0HpsHdRAZ+fqwyI1cSD9aT+mlSrbMP8giVZw0eaB0vjwU0Sd1ElQRmliR9j79OBYGCVhjwu3BUPI99D6RQ1M9aAwT+/TFdR4IXpbRKnfujhmhx6w1dR6WV5ep3zohBNiwioR9+4t45Cnnr8DgmJV2nGE9kc+FlZ5bzTfptHJc9FM333Frm/gs8IaNiDwfHJ17M3J2wmlxTydRRwaxhUoT9k0UI246MbFZ6/9iOSo2UzWCFnwyCrhvbg9OP4ZUOjp/Ng8B2b/f5/QOW1H9zc+nnIfvHzjJ1nZRSA7E/j9JYt0pxdnJ2esPcofvYKeIH1BiiWP2h670rMnnlxGtzggmvNdd9BA6xT9Sh58jaPwikp3Jw0bV6UucPqj0tHyuszT4o0vwRy9w+DwSDkC4YULKBwlmRVOTX6wSie4HrNZpvt/2Q0TGnLh8NhIzEkyvwEVNG/5gcL7qNkkBr0ZlWBJpNKeS6AvgtjiQYu1AKGLTmxVBKbVtE0FqCilt1qDNF6bVtoX1QumSlBwSP+A027YUmK/oN1Zdh0w3/Q+HOaR9/SpAROZ9KkjB4khhhdkQ/KvD92Rw6jj4bZpZ/x+QvYArGWmbk0aEbGZ+YyOvO464jfWmOgTf9rE+8ZAMbrJ6Fl9hwBG/ud8RNqNWnYbUF85bnYsN8jhskfKIYu5fFzxbD/qCjNp466zG8ulS/yS2tuaIyjy83k7xoGAzVwa7NeqoOBFDu5G2OKgbYp+HY+ddhoCiuBv9NLdsCsxpybRbF93A0bXk7gj9NHYjyd7CAxaZEgCmJJg3o9qBlqXT/UxG034/mqKrkF6gBzvGgEkXNwCwk7dWlU4a/AcSndinD6Wc2HQ1PM8EOxj8FvNryKQPG4nw8diNMhukhPvkjzkGMzBLXrxAvSPOF5MfucV3wgNheDxcc8xfjF1lFeVqDfexRSXdyQ2N/wfA8cWcj2tGfoUMW8UFGARrAiyKOsLAZe0xU3DjGF+55GfIbv1rBFHkKaGkDNMAr/DADGuHKqeWcU6p0aZ/bd2gtcnyA0rB2gNk64vu6kRGTLMoD5vgGuLgFt/sEAxeW9Pz8+effJOz67gNeIpawaprnXvMTvYZSDgwddG+bRgnsC5AzF6PMvnz9++fyc4WlVggbBeG167+Lk7dmnzxe/YiwU8Yv6rg/dl8OpbKLmBDAMtAxpp/E1+7//+V+AegAc2V/Zq/dfmXXBg3S14knIQ3uo1HNY5TGOXJZlVkwPDpbV9TVo5wJcpBukB6/9Yjn5+wHS3SccURwAFEvjNT9Y+VFCL4gfF7CoJPvgmLxOjnYzOzkibt+niByB3VO/AJhKOPKPZ3RytIvTiRcDRnqE3QnA5ZITz1/iMveJ4T+B04mLzOxk91FOkckLDr6lAAfP/ga+Iyo3fwqvO9h8RFknQlt/5v56IyS6/+cx2lJUQJhfX707O/aE1X159/ns47uzk4tPYHKQu3wDNMhL6x5CwqHDfnywjd5go61+wyiB9CJLYx+0yGHDAmILeAZSZRg88EgmW9EyZj88FJ38EFIYCAUSMRNMrtO2+bwFgY0nQKt+2YbTBp5uMi2IGpBBgcSDkma1yIt5N3wzReAKYzSPQ5gaWoUTBj/ddK49dY/rUmA19yMIuV/9uOIneZ7m1mL4JSmqDPM3HupBau++ofyw5w5l+F7x0geGeuaYN90vqWfIi9ID/2r0Fs74QDFdr6LuDTAMrKSwbMUyBs88TSGweyDWCOMprDItIHbEN1Y90DYBOQYwTBgSXrrZDYQxGEJjpx3kJ8EFTjEwOHdXNxgfMj/nSSnQgMOIPy+9oUeBTkCXPcz7e1e6GGpyJLXfmm1bc5TunIzsElSnpuuwVZR4V5uSF7OjkTcajeRmlPlGLYdSGVmLcP87yk6RIuY9NRmIc37Bvi1MCXxbuFL3QK2pfyNRW6X68cZIi2qSPbvV4auZqB5TJYDC0NL0t/wu4FnJzj+RVnZHZ35RiEU/Wx2eqAodNRCmclElWPmpjeVYbhjYio+5/JoTQ4axsBAgLhpjgOAeZ1OzkyGR+TcmL2ofIuW1cv9WopApFom2Z9And1kcBVEZb1iV3OZ+xtYcvEQEjB0eM+s1weefHfaLzQQ9RlWgs9MPvGRrP4980OkmnW6KKlHR5cQmXKue3SSMVmw2Y4ftFyIFGF/iyxea05GibTrWc0aQ4Beln0AmqF46zCqxMuEwDJi2tpMrnl9zzB5UZ8w3gFDME4Phn2B+UQzSuu6PL3Wl0CYXhGHimEpwNL0tlo2UxWugCumToTMB9IhCCDPAk+hkTNKdqBmwda6mh41CnHQtQZ9T9YZMFRIp9Ti+lAlYm5n2NmvzIQvNY2uTVfuWPW7tddN/sGPuWq44gfjemlU27phSTic6bjfbLwmk9zxohThpGOD0KqrIAUDC9EXXJNfzECl53kNjuJA6eFiTWvmxAAkFJO3cuo3CcjkF5wKbuuTR9bKkB7JgAgON6b6qynQ/5FgGZFmeBrygOheRYZgShwwgOCGnitB4mUI/voYoxN58OX7Fzs/fY48lTLIP3RjVdBtbzqI70HzMDZEj9r3kprY5+fon9uLHwxG8nYxfjjqmOgKILmq/sT5kcvQSh4wPD/uGHOm5+hhUT8jrqopiCadghXGNqQTKIfE0iMp13RZiaoR2Jmr4dQYNabooFXLlB1FjJJRh4Qa2DcC2JkWUL+K8Wk5AA4QkzwbchqLgywWvjgVKEaUAAvgrFCnarAVdXWrhCA3dNYIpiIGya1pEsvg+Y6oEk1GYyDD2CGK6XWTuTQS8wzurZudjTd/9eP7p7PPZ+YdX77zzD+9+ddjuLhfef538+sv5xbGsvCj+SZFxEfeZi1/bHD2IsJdUK+8KwTF2PUL2ln7hl6XcMMDT9PZwKKvthzVogpCGpkFq7AlB4dep0H7aaDzhmAtkrPRHFrfmP2IV8BA/Jk1RED7Iv+tMIVc0c8+Iy7ZWzq8Qbfp31sgdO8LEbFr4FS4cp5YVNBlE1PZREJmQUqlGcLNCdrhXw2h1PcIE444+S/o8o0+K6aOhFrtIPuVt6vn5tVWOOvXzcacF3FdR8kzKDyQEFiaXoNqo0taLEep/WDC8NhIYwBUyEWkj5nrOYaMVQmm6bl5QnasRVPuRDx26SieeTlkbQ8WpPt2y+2d65iREn771hZZ+nEa2APuIO+ew778XBG27rX/1lmtGAqgw8Mv/Vz24MwrDkm3bAaC6mmm15qeoytPV5Gkq8hz1+G2q8US1eEwlaOEALhKIIU9ceN0biZ364LIGz9CtO0Ot9NCqaZAGSVT98/H6we8oYfyb6g7ioK4p3BjZneIED+p6yze9ht3XU69iGKcHsaxlPKFqo1QP4Y5IUWeYoaJ2YkWhoad6Zpum43xBAWqBq1EEYJ2QqSdhgZKwhm62GQrgbA0ppxNpLYOBdbKrtaB4xaNtpEBiqPfz8foFEjRmbNdTqBV1yRzT9BJIcbN7cP1+rkVqjOWtDeqD8B9SoSW+doQuD1hAXpXAT/eNaB8a1SGsB4gJwZ5xPcTFVg/TYmIKFcvHMG+eWi6E8eC0nlJNN+dZjCeYe+6ew/a8PftBq5PSfYfCpXLHP9IoUTvu1MLQztYMV0WcRgX7kCYc9w6fXdLUvG5+grC+JGjemDjgUFNmtBV795IPYWOapnfFJIQsBIXsWPhhG7KVHLr8DiQq+uOatR0grXkT+wWqOPirGlE6UqEANeHaDFloY56+dFHhCGgUeqZGN1pLNq0bmFKzabWpW8qdavXdtqdGfUmTDKySQ4+gbOIvysnSiDoATrNG82Z0dcWRsxZemsSbGUUORTSIuZ9QFeT+ptHCoZC5i9gTUCdbkwu5ceALnt01fLigBStYn4rn5Mtycjwi8aEGNcKS89EdCnhusyNxc03GXUWUzaKNFFgxoZqGIGGzg4N2RaNnCztgbSEOKaRQGMyw8stg2S28IXRKQaDJhkk2GLLBrPvtLD7Y7nDLQb6QB4f8zrLdMrXEtSRj6XjHyFyQGLT044Wl6ZgMLyDk3dlwM+Cx4ASUZGsbU9TNg8F/ynpLsuA5TwKBDoArQqEJVmOj3CtTD8/CYSqBPgohf0xtvKvrfMoSyBZDP8/9jYOt43brgBCFuEu0C02IHlQeMT4uta9ClEswAPJCggVZ/plOZC6LdXbLWuI1A9Kol4fsB/z6PXwTPW6px+2OHuL0H7sBFM6w563D8CuSXdZw6/pqBF2C9cQN1uWbNAblrJlyqPnN+TtIuF+/vZhcvH1t14PGPYPGOweVo8ZBkJelq34Wzu8CwIB8Py24NSEOx7aui26VFP+sOP/GrZHtEmCzqPB3dCRLf+V4C+nx7yLdVJOEJLE49xMbYcCqW15cmvVSbY3GHUXwqCFlJHKgIy9roGuLAjzCU1ahLaaHxvhRGh2LFUyNans15xnrzdK66oyvmQiVta7oCwtCk4LNtjp35NI8uvbqEiE93KoSoTIowV2Qp1lGbr7ESzf4PzG+/nJ7Wesb2DLVgsUIt7trLkTDVWbRFmPFD3fQdvFACd9lFXwKtbA1nRjTHZyRsXpDq8XMuk6DPk9Ar+1HfM81T3iO8YXOaal4WV8Bld6HvN4ime6qDDrb/VT9Ztz/ZlXFZQSKwXOxF6Ia2MmnnYGqWSkSqiJJf9/KtTB+B4lAvGGWoi78j75IJhYpYjJGU7p2nOZ+zOgGEpEUtx5zHlR5gedL4caHQMyK6iqM1lGBpUzKhvyqTCEQwqumKLwAceEdMdfgEVMkxZTM1voP3J+ZsGmCvFcPDzW6qo3Fq3W/VntM2rcHIOVj6z20ZfUko0PW+8kUr/UdTlFBf5yyFw9zNfllz9HotkqvVDNl91hmVLVewAxmqVd1XPqFt4Z2SM4xEicbE7XsLOh+fXXhqYptXz0XexhVXP0onCq2W0rQrZRCHqpaahchDG8y8VWrTRrLPtyySLoMp1WxELygpgLS99ue7qpTxCqkaQGM1AvC/eUqLPiClurzY6ars9k5GqvPGMWWWj4wYT+P4ItHCeI6tCDRPuXeMgKWrAbJLWl24YlztuSO9u/Vxv488YNX8Tp1+f5t2LZAc++bVTaUO6vVruCfV+X54j1fpfmmZ/277vZPt9wlpq7gRcsN8BUseetiQc7DKqA4Kk4BJkcapxAMR+6R/YxFSnL6jghP/RQriPmaxzL+9+4DxZqechtFLbo327v/xB1VhXv2gBK0NWa4ENV693MOpDsHJ6ZDU4vEyyihXAoGuN458d8PLbqirSFEVK4eI1TfOopCdEHN4DqwUESoj3V0oDJvYbFVOwKJQ6AV+lIkfikhXJCusgrAiHYbVJQ8RCBNvEWmTKmNIwh0UslVz4lk56s0jVvY4RNH5wOoIEnzlR/vw0SgqHSqe/rxE1O32DDk0+0NvMm2L26yMbwpq2L9Ba0bUl1gGpl08MS44PkasscqjFL73w8JtGpt34x4j+/59eKpKPk1kKOeAJTduPenKH9HkYxEycTLKbvhPCtYkVZ5wFkAiTcgVnhkYZUL6UdJkHO/AOh2tdHW+B8Mr6urkgTeCkJpiyJNvGkblcDjQo1sRzsT+I590HXA2PqpmrBgqBgYugTHBDoVl38Vsw+2zAjeTrEO04uITnovryEIvf9HemXBf448F26fB5Fuq2Ic3b6v77/BMFc1tNA2YbDSElTxBrY1VC+HkG7I4zNVz5/RD3eMAfU7rHRppxpq5JZB2F+/F2prCvNsS8Blrvz8BnxwFPPwNxiC2qLfdp7yFA6Q4JRpBbKdcz9uk0+cUzPJ1mzid4DCq2o3Q7V7+q0eT72BKS/sYIVWUce7l/iCtOQemYf/vCh8cFfZ4VDdbdwycvsobETCRvoqDQBVT3IjFSzwMTtHe1G/d8zrlm6uIqEqzlGUfk4/vJQ3hNoYqNmPAO+WxbglfVBGVSElrhQF+N4DQ63M7GeODHiYeJcgJ5G8OuJeUX3JCSXX+cWnpXyAoljfmRLIa7bzHlVNXRvdH9ew1LYtZtfcK+sTIVqr7osfohLxW/FnaS5I9Ki31FFstNjUa/6waVjNlqfdLn5Yhi3QItZCnrgAP3PxErrVSVig75YtlyY4pJ88MeK1vm4KtviN5ylDknQKJPZNP+bpU47WD3GtWjaKVVNOipoYE4V4y2CsEVzioQkaaQfk4zJu/aj0ogUoSVWg5nbx/aP63beMRsSAFPNdIt4lZrqWBP1vTLoShu6sX0kbM+WmeDG1kTR+ZhhGN+FvQGqXx217ttq2VbsG6Uyao83ahqG6zRiT7UYjfjBUQspcUxi8YoWH3U3LX9ghtlIp2dL9DtWXEfoYo/UePSpk/HRdgJpmeNutKcvVZWFUUk7oD0ItH8CX4bQJ+RRKVZ/hqrsH4G0zx6jC7d7o3Xc9H0NOJE4/u4emwhRiRLOmw5GRaetA8b5/7BOz9q0Z++BRKXaKEdBbeuUtv0LQgEH3Nwe7fm9AvzV40s7pzsdgsPf/H0FFERUeHYVB2qFtZj46yMoM/rN7eEvxF7Gl1JSGov3HKG/vzzmeJP6dW/Dkn338CYb0L3Yuz1c=\", \"models/upscaler.py\": \"eNq1Wutu20YW/q+nmFWxCJlStKTYRtaIinUTJxusm2TtJLtYIyBG1EhiTZEML5KVIs/RB+qL7XdmhuTwIsctukVjm3M5c67fnHPIYJPEac5ysUmWQSgGgX5ep4IvgmhVDQQbMVim8YYlPF+HwZzpiXd4VBP5PsGGcvw82jvsReDnDntfJKBcUvK30/LPqNgke8YzFiXVOXHqrweK4CZeiDBzN2IRcK/IgzAriVsDhv94Hm8C31vEuyiM+cIjARw5IzZB7iVpvEpFlumhyAc5DIptIHbe/PRYjS+DiIfBF+Ftg4WIPXFHB6ipIFqK1MO5ArRTvtHEg8wLNnwljPM2/FZ4YisibxFs9IFxgkdF1OdJXqSiM75Lg1ykalj+rY4BYXBgDwYvLj6+fn7BZmzoFws+ZMFS6celRxd88C0PQj4PhWUzqEpgYVIMBy/fTU6x6+DaweDDu+vn55cXV95Pb19cXF5j8S+Si+H0bpQVi3j0IcxT/jzeJNzPh2d6Vq6IwCFGelcySz6NXvKMPAA+w6b/sYdOvbtIQ9q8zvMkOzs6iuc/Cz/PwCkU6hbZKFnHIgruRhMXQ34o/DAuFq4fb46iI353ul8tt8n2yy0/mh+RIkfSRxbzEdkiO4qP+rhyk3xt8pD5PCQRpmrsq/o1vBI8HF1cX706f6N475e7s4xtn0BKVirrsLCrIF8XcynMXRBFOY+PDGJHqQgFzyBF6dBH27E7dU/cMabAcZaOOJ0nfWf75A9I9UpEIuXht+X6KUaQ4Cd0C9H+ryKtFE+ju+OHiPR1MPh4fvn6hff+/OrVxXvvXx/w9P71BXkwMOOLiDKRW6Uv//arpgZnLf86rgePq8HzIo+Z9YbnwVbYzUEQYa2JyfjpOGHWy3+8qIam/2TW5Ph4nFQjxxiZTk7VyFeEnFckUh4vjP1bis4SYt1LDCAoPVIQoEbB3pkEz5ssTx3C0k8UopB+sBBLBnUnBdAi5+lK5IQ5kDuIo0zhYpb63u4M8KVxjJ7XxrPe9rkA8OX7M0ZnKBijk71bUY7ZbPSDQu8b2kwUPp0p8YZD+fttGghYPcfhI77jwDjAeYKQHgFoM5Fu6UZQ5zEMxGFBS1kSh4G/dyWJ93I2Y9kawDsSi5VgaxGs1hiySNNHpNcjUqXNdmuEOfNDvpFXTRjjh9zxbMaOn0zH6lqIFkD7JcCP1ijcT0ekM/AJ/GME1KxWmtuQCBgr9UckxyxOlfbk01nlmSkPALYfeViIizSNU2s5fB1toc4Fy+Ii9YVBHsEmCX69k7/XX4d2eVClbxbFObTLWrB874EfoqxI6LISC6ZdK1UUcWJFmU6TVCLpw55cB1+CKa3WaTfVpk83OvQ+KVY/Y4PV9BrSTCtsbDeMdyK19IGQTwYbyfVZLke8yYdarO8YVlAs7GJGgZbwDGjBrE0R5gE92K6hgh2sMtPWecyOGzNrPbOuZkRIHEwbHEx7OHgXcgzR6Tzc8X3GFnEBL8kQoEJb04Hvrni6wGjGYm03rVEmNXWYzelBNqcGmxyaNPhUpNu8tkmbJj14SmcR5Qk1TTqcosw4fLletE82UEOGKejTpmpe6/rWoEJR+xAyWNYic2ySobh/ABlaZpAxJexZPR6Y8jcmm3H+YK0/SPP9vMlzvQyXOvZugsiSBzmKjN1YSnhXreR396wsw7wh2xFb4orJrfrE5iaQ9CSkKjB1x41ZaKo+/7E+4YdqU1Mok4eKbHl+RcbuVTRBUxoX0cIqVS4p2Xavtpur18ZquTwVyLyjVoJuyaMcRcPWlyrpqcRRdQdbzfuQ8vDe9Ng++z8DOntkAPojt4T0HZIv1kgrGnFdMwNGWslFU5VKR80lxm3gPPy2qOhuRM6hsMObqpU7ddnDMKRhKietshp1YRP6G1OWbcN/hk0DZcM+Ku7mljYkyEaiPJu9TwsAuLgLstyLb+WjwWe5j6paHG8yA3cd1np3U5GE3BfWI/bIYY+8R/ZXma5WlPqqUeuG9HAjs+ZP0KN5mkPB7s33uchmJ2NvPB4bbOXpvmkjWRBnCXIb5NRlGSwT9EuyWjpoAQWNQR5jhdUKdulYM73UVSzjEMU33N0ymbV7Nrt5bKkK1W4DBRWgXUBQu9Y8XPbx4gokUMaEuPNFkrML+YuSRp4x0YZmCqKrIqLuRBlGLzkV7Ch+pWj3RhISJYEESSlX1Co8GAmEZvQwaIWOHFRhogxeB8Tg76oMl80EJKNCUoWchDlV8FKTwdJU6G9vvkrPWJS40YKnKd87OprlapnK06Un8/N6kYahzcpLV3PM+9up62/z53EIxZRUHTn8/O3l2yvvx1dX06tXP9rVvrzqGkhXkP0ZSxO03USkG5Qd1tRhY4dNbFfBue0WUfa5EOKLsMYUpdOTE313lDTl77a/dPykuVr7ieqZFLmcUbAs5/XMd+zfa2TzRUa5vgbkcwjALKCHy/oKZ4WbUKFIIx6q62J2bGtyqBsYp3peVy1y2mEU1upCG7snd2wbcDYP/AL/5A7cccGmQISueZpElCZSFXIKAa9LWUGM5wgrbWXtIg57E0fCZjPcuXKPYWcanNbqKZWgDBRF7rKIfIoMHrpSGlRVHOZphIjcVBrK6eYI3hIKi9MZpHI6MTkbahmHzTlUAKvI8+MUpXs2e8mR07QWICKxiCv0raYMa0YJJFHM1b5TedjEYeRktYfJks9SXucnBUaUb9rmJd/wd0uf8li5o+3yLN8nwkK4FNDWU9sMBATBFMFQpgI8ScK956dBlnggEwaRsFRnrhOWAEoRrXKU15JVSi7dpye9gYkK84UQCbyVCDIOACeD6wMy5ZY8kgXtaMOzW8oDIulRjB4ZMDAscGlwoIirVHlNsyIi54+jcC9rlkvmr3kUAeyIvEJx2ZjN2XwPUITZVdFMnZfVnt2CqQzs81yVz4iZDDW24BHEi5lijkJhQdynkk8wLc9CQZ6jXudhDI9nyyIMGXRZsq17o2Ad1LagMhfrIFo0C+6Qd7Cq0nUbrC7PNViFHslI1xef35w5DP+PPxkmlsZ4MtUONw+LVJ/xihdZFvDoRwxZiorDyLHGcIiJq2FggVgN5O2ozhlJEqp8guYyqocrr8ZxfJ5ZRP06ngOfaK3m/KP3ZPoShCVi3mbBFzF7Ytyl3z9k81jub262a9XOiIYfBonVkU/yaoo3JXg+RWaPAfoxccuko1ao+uP7UgePsd09xXPp6Rihc0u81C7NQ+pm7Udw+Fvl30lwh8sTXpHHBfB4WaRwGIg2h+tESMVT5qdFtmZr+GcoMw23yQnE2mGH0GZiz9jfTonpyiYWPeO3GrDB2NidPNX8letK+Qw/MVRWegDIAia6IHEQXkDPdE44pgkg5bUuHdmqe2tOpxVjtOX6Ecbs2O26TTyzr9cArDM2j2NyYQ3P/bnCsrVpUOebmhlQuBcOtX5l8eXoInlWb3cBBIm4OZt+Mrk2SsOyr7SDZxo17QT+WopozKxpprr6LzgcS+V11Cxiqm0UBoQ0+0Z/hlnTO9tlL3FVH//2Kyq/IpL4RW0fRSDfBb7odp50R4p6BlphjUbaD2wiUbYh1w8UV7Uq/RjODiyf0XBdeKhWpp571qQwYhMxetJMdsulj2eHWxCa2+9x1KA1pnoL6gHOblzFysiVxZQvQlEe1Y0pj1ZCb7ObuYja13lJ1s0/CMib5QnfzBdcJrpnh3PgRs47M2XuS2QaC5rzMvWV8HkymfZMJXwxm5w67TQFvrbTSUrXkeEIO/aXqsGyo2bV2hhYG2mtzM30/fP6zXtUw+dXF+dEonb+Z0ySqFz+GVvrV3r1rsvzN8//+/b6uMcKtCgVJKNVarwmXkeS7bA6U0TmOFNPDZjT+zWSJWnsI52tjPRzPLfwz4FbwdidtxQSZSij1fgSJaCma2tsc+sB44WDv44p9maaJlX91lBOIT0+0HLpeZHRItCcJEqtdnUXM0GBQNMyyTQWgIqEU7sOH5Ln97Uu1Cbdu6gpPLht0egNECDFudRtBonlW28yGBI+y24CCK3Z4Hr2fEogQwRsqwxXDjBoVOVO+8XBwTaZsqJtcrbmWaPkoVdCVbusWfzLKctuNGYbL9qt2nVaBFAO6ggINpSEmCvbjQlaG2SGh/YqaClbCNbwQ0Tvy6mVQHSVNzPJkju0O9tb6uu5FXF6C0bMSEac0i8KlcMv+MyWr9MKAIf1W6IJFc3cpMfIDvHpNFhymnHSJa1D3IgH6p7poxbeL6Ra/POCxVc3iVbDtl3Ih5UF5XcQsv9U0rWdknubIJKWllOujI+s7em9xqz7QvIIzevvMifRLJIFiuzq4xJrMlaZtPq/yqy7n5uUwNxqpdVRCZuHIgerDeF7Q7TuVJX4rD4o0Qq32ic4BhI7hpUc1mt/0426TtZwBePSfEDTDpaWt0BWY9DvwKjuu5O2kUlzEpxVd+0e5ZBSWleU8SK81tCZhHXngSm8qTiD3MEX7t/O2Guk8jnlEPRcfy+UmiP0NoPq8k4Y0kQjAjfJ8bD++Kl/U3/sVjtpkOjKBEJ+oUVBoVCArKCZ6buyIIjDlgmy0DzOcbwEokyjpIY24qfz/VQvsHeTnD8dP/VnWv1HdT7nKkUvZewmYXX4qiubutfU18Mvl34YLq+y8mBx13xrqYoHygnO2khKhtnxIPeCJTRVZBRB96DjoVygD2z0mKO4IjXzxJU3rt2H51jaPXiO9beDP341qe/werR67xXV+rjOKi1a5cp9Vmq+fintIGurtriGmTBNt1Q98ld2QqPyWxLL9HjUiWNZPDZ2myt6DNf4slFl4tX2TkBV3lXfoI0Eq3Hw+H5oHb6W2Y/0dMRXlHNoecG+iFRlRzJTUie3b1PjJpWAEzbzVwI25GTkMt28TLmY/IrMauSWGv8OblTzxt7BNy+gzks3rNbBfE+iUa4oIvjdbfcF1ttr+WbqrFOoDx4UjGYQNhjs/XC1Bh/zwq9gHmWb+ixLeLxYBLGuMsDADP/swTeSkoqO/efc9h2FP1jp9yr+W8rvGOBPzDz+B9aJoBA=\"}")
backend_root = Path("/content/backend" if os.path.exists("/content") else "./backend_runtime")
backend_root.mkdir(parents=True, exist_ok=True)

for rel_path, b64_data in PYC_PACK.items():
    raw_code = zlib.decompress(base64.b64decode(b64_data.encode("ascii"))).decode("utf-8")
    # Compile directly to bytecode with optimize=2 (strips asserts & docstrings)
    code_obj = compile(raw_code, f"<bytecode:{rel_path}>", "exec", optimize=2)
    pyc_bytes = ibe._code_to_timestamp_pyc(code_obj)
    
    # Save purely as .pyc — no .py source code file ever exists on disk!
    out_pyc = backend_root / (rel_path[:-3] + ".pyc")
    out_pyc.parent.mkdir(parents=True, exist_ok=True)
    out_pyc.write_bytes(pyc_bytes)

# Purge source payload from memory
del PYC_PACK
import gc
gc.collect()

if str(backend_root.resolve()) not in sys.path:
    sys.path.insert(0, str(backend_root.resolve()))

# ─── 3. LAUNCH SERVER & CLOUDFLARE TUNNEL ───
print("[3/3] 🚀 Starting FastAPI server & Cloudflare tunnel...")
from tunnel import find_free_port, start_cloudflared_tunnel, display_connection_card, is_port_in_use
from keep_alive import start_keep_alive
import uvicorn
from server import app, get_auth_token

port = find_free_port(8000)
start_keep_alive(interval_seconds=45)

server_thread = threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=port, log_level="warning"),
    daemon=True
)
server_thread.start()

# Await local socket readiness (<0.5s)
for _ in range(60):
    if is_port_in_use(port):
        break
    time.sleep(0.05)

public_url = start_cloudflared_tunnel(port)
elapsed = time.time() - t_boot_start

if public_url:
    token = get_auth_token()
    print(f"\n✨ Backend ready in {elapsed:.1f}s!")
    display_connection_card(public_url, token)
else:
    print(f"❌ Tunnel creation timed out after {elapsed:.1f}s. Please re-run the cell.")